In [ ]:
#%% %PLOTTING THE RESULTS%%
window_size = 200  # (for smoothing the curves in the plots)
#####
LC.visualize_ru_du_locations(du_ru_adj_matrix)
# %%%%RUNTIME DURATION%%%%%%%
# Calculate the average episode runtime and its moving average
# mean_mat_episode_runtime = moving_average(1000*np.average(mat_episode_runtime, axis=1), window_size) 

# Plot the data using your custom plot function
plot_graph("Runtime Duration",
           [moving_average(1000*np.average(mat_episode_runtime, axis=1), window_size)],
           ['Average runtime duration: {:.2f} ms'.format(1000 * np.average(mat_episode_runtime))],
           ['blue'],
           ['solid'],
           "Episode",
           "Runtime duration (ms)")
#PRB Allocation?
# Calculate the average number of PRBs used per BS for SAC and SAC_pred
# avg_prbs_sac = mat_used_prbs_per_user_per_bs.mean(axis=(0,2,3))
# # avg_prbs_sac_pred = mat_used_prbs_per_user_per_bs_pred.mean(axis=(0,2,3))

# # Create an array with the positions of each bar on the x-axis
# barWidth = 0.3
# r1 = np.arange(len(avg_prbs_sac))
# r2 = [x + barWidth for x in r1]

# # Create the bar chart
# plt.bar(r1, avg_prbs_sac, color='b', width=barWidth, edgecolor='grey', label='SAC')
# # plt.bar(r2, avg_prbs_sac_pred, color='r', width=barWidth, edgecolor='grey', label='SAC_pred')

# # Add xticks on the middle of the group bars
# plt.xlabel('BS', fontweight='bold')
# plt.xticks([r + barWidth/2 for r in range(BS_NO)], range(1, BS_NO+1))

# plt.ylabel('Average number of PRBs used')
# plt.legend()

# # Show the plot
# plt.show()

########################
# Select data for BS=0
# prbs_per_user_per_bs_0 = mat_used_prbs_per_user_per_bs[:, 0, :, :]
# # prbs_per_user_per_bs_pred_0 = mat_used_prbs_per_user_per_bs_pred[:, 4, :, :]

# # Sum over users and then calculate averages over Monte Carlo runs
# sum_prbs_per_user_per_bs_0 = np.sum(prbs_per_user_per_bs_0, axis=1)
# # sum_prbs_per_user_per_bs_pred_0 = np.sum(prbs_per_user_per_bs_pred_0, axis=1)

# avg_prbs_per_user_per_bs_0 = np.mean(sum_prbs_per_user_per_bs_0, axis=0)
# # avg_prbs_per_user_per_bs_pred_0 = np.mean(sum_prbs_per_user_per_bs_pred_0, axis=0)

# # Plot the averages using your function
# plot_graph('Overall PRBs used for BS=4 for SAC algorithm',
#            [avg_prbs_per_user_per_bs_0],
#            ['SAC'],
#            ['b'],
#            ['-'],
#            'T',
#            'Overall PRBs used in BS 4')
# plot_graph('Overall PRBs used for BS=4 for SAC and SAC_pred algorithms',
#            [avg_prbs_per_user_per_bs_0, avg_prbs_per_user_per_bs_pred_0],
#            ['SAC', 'SAC_pred'],
#            ['b', 'r'],
#            ['-', '--'],
#            'T',
#            'Overall PRBs used in BS 4')
#######################
# Calculate averages over Monte Carlo runs and users
avg_prbs_per_user = np.mean(mat_used_prbs_per_user, axis=(1,2))
# avg_prbs_per_user = np.mean(mat_used_prbs_per_user, axis=(0,1))
# avg_prbs_per_user_pred = np.mean(mat_used_prbs_per_user_pred, axis=(0,1))

# Plot the averages using your function
plot_graph('Avg PRBs used per user for SAC algorithm',
           [moving_average(avg_prbs_per_user , window_size)],
           ['SAC'],
           ['b'],
           ['-'],
           'T',
           'Average PRBs used per user')
# plot_graph('Avg PRBs used per user for SAC and SAC_pred algorithms',
#            [avg_prbs_per_user, avg_prbs_per_user_pred],
#            ['SAC', 'SAC_pred'],
#            ['b', 'r'],
#            ['-', '--'],
#            'T',
#            'Average PRBs used per user')
# Fairness for PRBs
# Calculate PRB utilization for each user at each time step and for each MC run
prb_utilization = np.sum(mat_rho, axis=(2, 1))  # Sum over PRBs and BSs

# Calculate proportional fairness score for each time step
fairness_scores = np.zeros((E, T))

for e in range(E):
    for t in range(T):
        # Calculate Jain's fairness index for time step t in MC run mc
        sum_of_prbs = np.sum(prb_utilization[e, :, t])
        sum_of_squares = np.sum(prb_utilization[e, :, t] ** 2)
        # Handle potential division by zero or NaN
        if sum_of_prbs == 0 or np.isnan(sum_of_squares):
            fairness_scores[e, t] = 0  # Set fairness score to 0
        else:
            fairness_scores[e, t] = (sum_of_prbs ** 2) / (BS_NO * sum_of_squares)

# Average fairness scores over all MC runs
# avg_fairness_scores = np.mean(fairness_scores, axis=0)
avg_fairness_scores = np.mean(fairness_scores, axis=1)
normalized_fairness_scores = (avg_fairness_scores - np.min(avg_fairness_scores)) / (np.max(avg_fairness_scores) - np.min(avg_fairness_scores))


plot_graph("Jain's fairness index for PRB allocation",
           [moving_average(normalized_fairness_scores , window_size)],
           ['SAC'],
           ['blue'],
           ['solid'],
           "Episode",
           "Mean fairness score")

# %%REWARD%%%%
#mat_reward_average_over_m = np.average(mat_reward, axis=1) #axis=0
# mat_reward_average_over_m_pred = np.average(mat_reward_pred, axis=0)

# mean_ep_rewardall = moving_average(mat_reward_average_over_m, window_size)
# mean_ep_rewardall_pred = moving_average(mat_reward_average_over_m_pred, window_size)
plot_graph("Mean episodic rewards",
           [moving_average(np.average(mat_reward, axis=1), window_size)],
           ['SAC'],
           ['blue'],
           ['solid'],
           "Episode",
           "Mean episodic rewards")
# plot_graph("Mean episodic rewards",
#            [mean_ep_rewardall, mean_ep_rewardall_pred],
#            ['SAC', 'Proactive SAC'],
#            ['blue', 'green'],
#            ['solid', 'dotted'],
#            "Episode",
#            "Mean episodic rewards")
# %%CONSTRAINT SATISFACTION%
# mean_ep_prb_const = moving_average(np.average(mat_satisfied_prb_constraint, axis=1), window_size)
# # mean_ep_prb_const_pred = moving_average(np.average(mat_satisfied_prb_constraint_pred, axis=0), window_size)
# mean_ep_power_const = moving_average(np.average(mat_satisfied_power_constraint, axis=1), window_size)
# # mean_ep_power_const_pred = moving_average(np.average(mat_satisfied_power_constraint_pred, axis=0), window_size)
# mean_ep_delay_const = moving_average(np.average(mat_satisfied_delay_constraint, axis=1), window_size)
# # mean_ep_delay_const_pred = moving_average(np.average(mat_satisfied_delay_constraint_pred, axis=0), window_size)
# mean_ep_rate_const = moving_average(np.average(mat_satisfied_rate_constraint, axis=1), window_size)

plot_graph("Constraint Satisfaction",
           [moving_average(np.average(mat_satisfied_prb_constraint, axis=1), window_size), 
            moving_average(np.average(mat_satisfied_power_constraint, axis=1), window_size),
            moving_average(np.average(mat_satisfied_delay_constraint, axis=1), window_size),
            moving_average(np.average(mat_satisfied_rate_constraint, axis=1), window_size)],
           ['PRB (SAC)',
            'Power (SAC)',
            'Delay (SAC)',
            'Rate (SAC)'],
           ['red', 'blue', 'green', 'orange'],
           ['solid', 'solid', 'solid', 'solid'],
           "Episode",
           "Constraint Satisfaction Rate")

# plot_graph("Constraint Satisfaction",
#            [mean_ep_prb_const, mean_ep_prb_const_pred,
#             mean_ep_power_const, mean_ep_power_const_pred,
#             mean_ep_delay_const, mean_ep_delay_const_pred],
#            ['PRB (SAC)', 'PRB (SAC_pred)',
#             'Power (SAC)', 'Power (SAC_pred)',
#             'Delay (SAC)', 'Delay (SAC_pred)'],
#            ['red', 'red', 'blue', 'blue', 'green', 'green'],
#            ['solid', 'dotted', 'solid', 'dotted', 'solid', 'dotted'],
#            "Episode",
#            "Constraint Satisfaction")
# %%%%%SSL%%%%%%
# mean_ep_ssl_rate = moving_average(np.average(mat_ssl_rate, axis=1), window_size)
# mean_ep_ssl_delay = moving_average(np.average(mat_ssl_delay, axis=1), window_size)
# mean_ep_ssl = moving_average(np.average(mat_ssl, axis=1), window_size)
# mean_ep_ssl_rate_pred = moving_average(np.average(mat_ssl_rate_pred, axis=0), window_size)
# mean_ep_ssl_delay_pred = moving_average(np.average(mat_ssl_delay_pred, axis=0), window_size)
# mean_ep_ssl_pred = moving_average(np.average(mat_ssl_pred, axis=0), window_size)

# plot_graph("SSL Metrics",
#            [mean_ep_ssl_rate, mean_ep_ssl_rate_pred,
#             mean_ep_ssl_delay, mean_ep_ssl_delay_pred,
#             mean_ep_ssl, mean_ep_ssl_pred],
#            ['Rate (SAC)', 'Rate (SAC_pred)',
#             'Delay (SAC)', 'Delay (SAC_pred)',
#             'SSL (SAC)', 'SSL (SAC_pred)'],
#            ['blue', 'blue', 'green', 'green', 'orange', 'orange'],
#            ['solid', 'dotted', 'solid', 'dotted', 'solid', 'dotted'],
#            "Episode",
#            "SSL Metrics")
plot_graph("SSL Metrics",
           [moving_average(np.average(mat_ssl_rate, axis=1), window_size),
            moving_average(np.average(mat_ssl_delay, axis=1), window_size),
            moving_average(np.average(mat_ssl, axis=1), window_size)],
           ['Rate (SAC)',
            'Delay (SAC)',
            'SSL (SAC)'],
           ['blue', 'green', 'orange'],
           ['solid', 'solid', 'solid'],
           "Episode",
           "SSL Metrics")
# %%%%%Delay%%%%%%
# Calculate mean delay over users for SAC
mean_delay_sac = np.mean(np.mean(monte_mat_delay_tot, axis=1), axis=1)
#mean_delay_sac = np.mean(np.mean(monte_mat_delay_tot, axis=1), axis=0)
# Calculate mean delay over users for SAC_pred
# mean_delay_sac_pred = np.mean(np.mean(monte_mat_delay_tot_pred, axis=1), axis=0)

# Apply moving average to smooth the curves
# mean_delay_sac_smoothed = moving_average(mean_delay_sac, window_size)
# mean_delay_sac_pred_smoothed = moving_average(mean_delay_sac_pred, window_size)

# Plot the comparison graph
plot_graph("Comparison of Average E2E Delay (SAC)",
           [moving_average(mean_delay_sac , window_size)],
           ['SAC'],
           ['blue'],
           ['solid'],
           "Timestep",
           "Average E2E Delay (ms)")
# plot_graph("Comparison of Average E2E Delay (SAC vs. SAC_pred)",
#            [mean_delay_sac_smoothed, mean_delay_sac_pred_smoothed],
#            ['SAC', 'SAC_pred'],
#            ['blue', 'green'],
#            ['solid', 'solid'],
#            "Timestep",
#            "Average E2E Delay (ms)")
#####################
# mean_rate_sac = np.mean(np.mean(shannon, axis=1), axis=0)
mean_rate_sac = np.mean(np.mean(shannon, axis=1), axis=1)
# mean_rate_sac_smoothed = moving_average(mean_rate_sac, window_size)
plot_graph("Average Data Rate (SAC)",
           [moving_average(mean_rate_sac , window_size)],
           ['SAC'],
           ['blue'],
           ['solid'],
           "Timestep",
           "Average Data Rate (Mbps)")

#####################
mean_power_sac = np.mean(np.mean(mat_power, axis=1), axis=1)
# mean_power_sac_smoothed = moving_average(mean_power_sac, window_size)
plot_graph("Average of total allocated power to each user (SAC)",
           [moving_average(mean_power_sac , window_size)],
           ['SAC'],
           ['blue'],
           ['solid'],
           "Timestep",
           "Average of total allocated power to each user (W)")

###
average_rate = np.mean(shannon, axis=2)

for user_idx in range(USER_NO):
    plt.plot(moving_average(range(E), window_size), moving_average(average_rate[:,user_idx], window_size), label=f'User {user_idx+1}')
plt.xlabel('E')
plt.ylabel('Rate (Mbps)')
plt.title('Average rate for individual users')
plt.legend()
plt.show()

###
average_rate_ssl_u = np.mean(mat_fittingness_u_rate, axis=2)

for user_idx in range(USER_NO):
    plt.plot(moving_average(range(E), window_size), moving_average(average_rate_ssl_u[:,user_idx], window_size), label=f'User {user_idx+1}')
plt.xlabel('E')
plt.ylabel('SSL_rate_u')
plt.title('Normalized rate satisfaction')
plt.legend()
plt.show()

###
average_delay_ssl_u = np.mean(mat_fittingness_u_delay, axis=2)

for user_idx in range(USER_NO):
    plt.plot(moving_average(range(E), window_size), moving_average(average_delay_ssl_u[:,user_idx], window_size), label=f'User {user_idx+1}')
plt.xlabel('E')
plt.ylabel('SSL_delay_u')
plt.title('Normalized delay satisfaction')
plt.legend()
plt.show()
###
plot_graph("Sum of handovers (each episode)",
           [moving_average(np.sum(mat_count_handovers, axis=1) , window_size)],
           ['SAC'],
           ['blue'],
           ['solid'],
           "Timestep",
           "Sum of handovers (each episode)")

###
plot_graph("Average No. of handovers per user",
           [moving_average(np.average(mat_count_handovers, axis=1) , window_size)],
           ['SAC'],
           ['blue'],
           ['solid'],
           "Timestep",
           "Average No. of handovers per user")
# ###

for user_idx in range(USER_NO):
    plt.plot(moving_average(range(E), window_size), moving_average(mat_count_handovers[:,user_idx], window_size), label=f'User {user_idx+1}')
plt.xlabel('E')
plt.ylabel('No. of handovers per user')
plt.title('No. of handovers per user')
plt.legend()
plt.show()


print(style.UNDERLINE + "Total time for {} timesteps ({} users) in {} Episdoes (aka iterations or epochs): {}".format(T, USER_NO, E, convert_seconds(np.sum(mat_episode_runtime))))

